# Relate entities of two databases

A relation is a fact of the model: it says the two entities are related and promises no foreign key, no join and no shared transaction. So it may join entities of any resources, even resources of two databases, and the model accepts it.

## Import the types

In [ ]:
!pip install -q aoa-action-machine

In [ ]:
import json
from typing import Annotated

from pydantic import Field

from aoa.action_machine.domain import AssociationOne, BaseCoreEntity, NoInverse
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.graph.node_graph_coordinator_factory import create_node_graph_coordinator
from aoa.action_machine.intents.entity import entity
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.resources import BaseDataSource, BaseStorage

## Declare the model

In [ ]:
class StoreDomain(BaseDomain):
    """Group the shop model declarations."""

    name = "store"
    description = "Store"


class SupportDomain(BaseDomain):
    """Group the support desk declarations."""

    name = "support"
    description = "Support"


@meta(description="Storefront database")
class StorefrontDataSource(BaseDataSource):
    """The database the orders live in."""


@meta(description="Support desk database")
class SupportDataSource(BaseDataSource):
    """Another database: the one the support tickets live in."""


@meta(description="Primary relational store", domain=StoreDomain)
class StorefrontDatabase(BaseStorage[StorefrontDataSource]):
    """The orders, in the storefront database."""

    def get_wrapper_class(self):
        """No wrapper: this example never opens the resource."""
        return None


@meta(description="Support desk store", domain=SupportDomain)
class SupportDatabase(BaseStorage[SupportDataSource]):
    """The support tickets, in the support desk database."""

    def get_wrapper_class(self):
        """No wrapper: this example never opens the resource."""
        return None


@entity(description="Customer order", resource=StorefrontDatabase)
class SalesOrderCoreEntity(BaseCoreEntity):
    """A customer's order."""

    id: str = Field(description="Order identifier")


@entity(description="Support ticket", resource=SupportDatabase)
class SupportTicketCoreEntity(BaseCoreEntity):
    """A ticket about an order of the other database."""

    id: str = Field(description="Ticket identifier")
    order: Annotated[AssociationOne[SalesOrderCoreEntity], NoInverse()] = Field(description="Order")


SupportTicketCoreEntity.model_rebuild()

## Run this case

In [ ]:
graph = json.loads(create_node_graph_coordinator().to_json())
short = {node["id"]: node["label"] for node in graph["nodes"]}
for edge in graph["edges"]:
    if edge["type"] in ("resource_data_source", "entity_relation"):
        print(edge["type"] + ":", short[edge["source_id"]], "->", short[edge["target_id"]])